In [1]:
pip install dash plotly paho-mqtt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import threading
from collections import deque
from datetime import datetime

import paho.mqtt.client as mqtt
import plotly.graph_objects as go
from dash import Dash, Input, Output, dcc, html
from plotly.subplots import make_subplots


BROKER_HOST = "broker.hivemq.com"
BROKER_PORT = 1883

TOPIC_DASAR = "uy3^*y1y3bhgasGSjas/2026/okt/sic2/temennyatimsebelah/esp32"


INTERVAL_MS = 1000   
MAKS_DATA = 500     

SERIES = [
    ("Humidity",    "humidity",    "%",   "#007aff"),
    ("Temperature", "temperature", "°C",  "#ff9500"),
    ("Gas",         "gas",         "ppm", "#af52de"),
]


kunci_data = threading.Lock()
waktu = {k: deque(maxlen=MAKS_DATA) for _, k, _, _ in SERIES}
nilai = {k: deque(maxlen=MAKS_DATA) for _, k, _, _ in SERIES}



def on_connect(client, userdata, flags, rc, properties=None):
    if rc == 0:
        print("Berhasil terhubung ke Public HiveMQ!")
        client.subscribe(TOPIC_DASAR + "/#")  
        print(f"Mendengarkan data pada topik: {TOPIC_DASAR}/#")
    else:
        print(f"Koneksi gagal, kode: {rc}")


def baca_angka(payload):
    teks = payload.decode("utf-8", "replace").strip()
    try:
        return float(teks)
    except ValueError:
        data = json.loads(teks)  
        return float(data["value"] if isinstance(data, dict) else data)


def on_message(client, userdata, msg):
    kunci = msg.topic.rsplit("/", 1)[-1]  
    if kunci not in nilai:
        print(f"Topik tidak dikenal, dilewati: {msg.topic}")
        return
    try:
        angka = baca_angka(msg.payload)
    except Exception:
        print(f"Payload tidak valid di {msg.topic}: {msg.payload.decode('utf-8', 'replace')}")
        return

    sekarang = datetime.now()
    with kunci_data:
        waktu[kunci].append(sekarang)
        nilai[kunci].append(angka)
    print(f"[{sekarang:%H:%M:%S}] {kunci}: {angka}")


def mulai_mqtt():
    try:  
        client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    except AttributeError:
        client = mqtt.Client()
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER_HOST, BROKER_PORT, 60)
    client.loop_start() 
    return client


def buat_figure(t, v):
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=[f"{n} ({u})" for n, _, u, _ in SERIES],
    )
    for i, (nama, kunci, satuan, warna) in enumerate(SERIES, start=1):
        fig.add_trace(
            go.Scatter(
                x=t[kunci], y=v[kunci], mode="lines", name=nama,
                line=dict(color=warna, width=2),
                hovertemplate=f"{nama}: %{{y:.1f}} {satuan}<extra></extra>",
            ),
            row=i, col=1,
        )
        fig.update_yaxes(title_text=satuan, row=i, col=1)

    fig.update_xaxes(tickformat="%H:%M:%S", row=3, col=1, title_text="Waktu")
    fig.update_layout(
        height=750, hovermode="x unified", showlegend=False,
        template="plotly_white", margin=dict(t=40, b=40),
        uirevision="tetap",  
    )
    return fig


app = Dash(__name__)
app.layout = html.Div(
    style={"maxWidth": "1000px", "margin": "0 auto", "fontFamily": "system-ui, sans-serif"},
    children=[
        html.H2("Sensor readings"),
        html.Div(id="info", style={"color": "#6e6e73", "marginBottom": "8px"}),
        dcc.RadioItems(
            id="jendela",
            options=[
                {"label": " 30 data", "value": 30},
                {"label": " 60 data", "value": 60},
                {"label": " Semua", "value": 0},
            ],
            value=0, inline=True, inputStyle={"marginLeft": "12px"},
        ),
        dcc.Graph(id="grafik"),
        dcc.Interval(id="tick", interval=INTERVAL_MS),
    ],
)


@app.callback(
    Output("grafik", "figure"),
    Output("info", "children"),
    Input("tick", "n_intervals"),
    Input("jendela", "value"),
)
def perbarui(_, jendela):
    with kunci_data:  
        t = {k: list(waktu[k]) for _, k, _, _ in SERIES}
        v = {k: list(nilai[k]) for _, k, _, _ in SERIES}

    fig = buat_figure(t, v)
    ada = [k for k in t if t[k]]
    if not ada:
        return fig, "Menunggu data dari HiveMQ..."

    if jendela:  
        acuan = max(ada, key=lambda k: len(t[k]))
        awal = t[acuan][max(0, len(t[acuan]) - jendela)]
        akhir = max(t[k][-1] for k in ada)
        fig.update_xaxes(range=[awal, akhir])

    belum = [k for k in t if not t[k]]
    info = f"Data terakhir {max(t[k][-1] for k in ada):%H:%M:%S} · " + ", ".join(
        f"{k}: {len(t[k])}" for k in t
    )
    if belum:
        info += f" · belum ada data: {', '.join(belum)}"
    return fig, info


if __name__ == "__main__":
    mulai_mqtt()
    try:  
        import google.colab  
        app.run(debug=False, jupyter_mode="inline", jupyter_height=900)
    except ImportError:
        app.run(debug=False)  